# Download Historical Data

Fetches OHLCV trendbar data from the cTrader Open API and saves it to `data/`.

`Learn.data.fetch_ohlcv()` is the single-symbol helper. For multi-symbol downloads in one run, use `Learn.data.fetch_ohlcv_bulk()`.

> **Note:** The Twisted reactor can only run once per kernel session.  
> If you need to re-download, **restart the kernel** before re-running.

In [1]:
# -- Configuration -----------------------------------------------------------
symbol_name     = "US500"   # Symbol name (must match broker exactly)
num_chunks      = 52 * 10   # Number of time chunks to fetch
weeks_per_chunk = 1         # Width of each chunk in weeks
period_str      = "M1"      # Bar period: M1 M2 M3 M4 M5 M10 M15 M30 H1 H4 H12 D1 W1 MN1

In [2]:
import pathlib
import pandas as pd

from Learn.data import fetch_ohlcv, fetch_ohlcv_bulk

df = fetch_ohlcv(
    symbol_name     = symbol_name,
    num_chunks      = num_chunks,
    weeks_per_chunk = weeks_per_chunk,
    period_str      = period_str,
    save_csv        = False,
)

:0: UserWarning: You do not have a working installation of the service_identity module: 'No module named 'service_identity''.  Please install it from <https://pypi.python.org/pypi/service_identity> and make sure all of its dependencies are satisfied.  Without the service_identity module, Twisted can perform only rudimentary TLS client hostname verification.  Many valid certificate/hostname mappings may be rejected.



Connected

Application authenticated

Account authenticated

Symbols received

Fetching symbol 1/1: US500

Fetched US500 chunk 1/520, bars: 6885

Fetched US500 chunk 2/520, bars: 6876

Fetched US500 chunk 3/520, bars: 6891

Fetched US500 chunk 4/520, bars: 6432

Fetched US500 chunk 5/520, bars: 6882

Fetched US500 chunk 6/520, bars: 6882

Fetched US500 chunk 7/520, bars: 6891

Fetched US500 chunk 8/520, bars: 6952

Fetched US500 chunk 9/520, bars: 6889

Fetched US500 chunk 10/520, bars: 6888

Fetched US500 chunk 11/520, bars: 6647

Fetched US500 chunk 12/520, bars: 6891

Fetched US500 chunk 13/520, bars: 6877

Fetched US500 chunk 14/520, bars: 6890

Fetched US500 chunk 15/520, bars: 6627

Fetched US500 chunk 16/520, bars: 6837

Fetched US500 chunk 17/520, bars: 5333

Fetched US500 chunk 18/520, bars: 5210

Fetched US500 chunk 19/520, bars: 6883

Fetched US500 chunk 20/520, bars: 6882

Fetched US500 chunk 21/520, bars: 6850

Fetched US500 chunk 22/520, bars: 5773

Fetched US500 chunk 2

## Validation

In [3]:
# -- Basic info --------------------------------------------------------------
print(f"Shape      : {df.shape}")
print(f"Date range : {df['Time'].min()}  ->  {df['Time'].max()}")
print(f"Dtypes     :\n{df.dtypes}")
df.head(3)

Shape      : (3219838, 6)
Date range : 2016-05-09 23:34:00+00:00  ->  2026-04-27 23:28:00+00:00
Dtypes     :
Time      datetime64[us, UTC]
Open                  float64
High                  float64
Low                   float64
Close                 float64
Volume                  int64
dtype: object


,Time,Open,High,Low,Close,Volume
0,2016-05-09 23:34:00+00:00,2055.5,2055.6,2055.5,2055.6,4
1,2016-05-09 23:37:00+00:00,2055.8,2055.8,2055.8,2055.8,2
2,2016-05-09 23:40:00+00:00,2055.5,2055.6,2055.5,2055.6,4


In [4]:
# -- Null check --------------------------------------------------------------
null_counts = df.isnull().sum()
print("Null counts per column:")
print(null_counts)
assert null_counts.sum() == 0, "Unexpected nulls detected -- check fetch output."

Null counts per column:
Time      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64


In [5]:
# -- Duplicate timestamps ----------------------------------------------------
n_dupes = df.duplicated(subset='Time').sum()
print(f"Duplicate timestamps : {n_dupes}")
assert n_dupes == 0, "Duplicate timestamps found -- check fetch output."

Duplicate timestamps : 0


In [6]:
# -- OHLC sanity -------------------------------------------------------------
ohlc_violations = (
    (df['High'] < df['Low']).sum()   +
    (df['High'] < df['Open']).sum()  +
    (df['High'] < df['Close']).sum() +
    (df['Low']  > df['Open']).sum()  +
    (df['Low']  > df['Close']).sum()
)
print(f"OHLC violations : {ohlc_violations}")
assert ohlc_violations == 0, "OHLC relationship violations detected."

OHLC violations : 0


In [7]:
# -- Gap analysis ------------------------------------------------------------
PERIOD_MINUTES = {
    'M1': 1, 'M2': 2, 'M3': 3, 'M4': 4, 'M5': 5, 'M10': 10,
    'M15': 15, 'M30': 30, 'H1': 60, 'H4': 240, 'H12': 720,
    'D1': 1440, 'W1': 10080,
}
if period_str in PERIOD_MINUTES:
    expected_delta = pd.Timedelta(minutes=PERIOD_MINUTES[period_str])
    times = pd.to_datetime(df['Time'])
    deltas = times.diff().dropna()
    gaps = deltas[deltas > expected_delta * 1.5]
    print(f"Gaps > 1.5x bar period ({expected_delta}): {len(gaps)}")
    if len(gaps) > 0:
        print(gaps.sort_values(ascending=False).head(10))
else:
    print(f"Gap analysis skipped for period '{period_str}'.")

Gaps > 1.5x bar period (0 days 00:01:00): 170287
1354436   3 days 04:46:00
162942    3 days 01:52:00
160005    3 days 01:51:00
383730    3 days 01:46:00
386342    3 days 01:46:00
2410400   3 days 01:36:00
2415257   3 days 01:36:00
2072660   3 days 01:06:00
2067113   3 days 01:06:00
460315    3 days 01:03:00
Name: Time, dtype: timedelta64[us]


In [8]:
# -- Price distribution ------------------------------------------------------
df[['Open', 'High', 'Low', 'Close', 'Volume']].describe().round(6)

,Open,High,Low,Close,Volume
count,3.219838e+06,3.219838e+06,3.219838e+06,3.219838e+06,3.219838e+06
mean,4.047096e+03,4.047655e+03,4.046533e+03,4.047097e+03,4.588217e+01
std,1.340409e+03,1.340551e+03,1.340263e+03,1.340410e+03,5.674662e+01
min,1.991300e+03,1.991500e+03,1.990800e+03,1.991000e+03,1.000000e+00
25%,2.857500e+03,2.857900e+03,2.857100e+03,2.857500e+03,1.000000e+01
50%,3.959400e+03,3.960100e+03,3.958800e+03,3.959400e+03,2.600000e+01
75%,4.773000e+03,4.773500e+03,4.772500e+03,4.773000e+03,6.000000e+01
max,7.193600e+03,7.194600e+03,7.193100e+03,7.193600e+03,1.424000e+03


## Save

In [9]:
total_weeks = num_chunks * weeks_per_chunk
output_path = pathlib.Path("../data") / f"{symbol_name}_{period_str}_{total_weeks}weeks.csv"
output_path = output_path.resolve()

df.to_csv(output_path, index=False)
print(f"Saved {len(df):,} rows -> {output_path}")

Saved 3,219,838 rows -> C:\Users\Stuart\Desktop\Trading Bot\MLQ5-Production\data\US500_M1_520weeks.csv
